# Learned Primal-Dual: Sparse-View CT Reconstruction
This notebook demonstrates the **Learned Primal-Dual (LPD)** algorithm, a state-of-the-art unrolled optimization architecture for solving inverse problems in medical imaging.

**Key Features:**
- Native JAX/Flax implementation.
- End-to-end training on synthetic ellipse phantoms.
- High-quality visualization of sinograms and reconstructions.

### Installation (Colab)
If you are running this in Google Colab, run the cell below to install the `lpd_jax` library.

> ⚡ **Tip:** For best performance, go to **Runtime → Change runtime type** and select **GPU** (T4 or better).

In [ ]:
!git clone https://github.com/Forgotten/learned_primal_dual.git 2>/dev/null || echo 'Repo already cloned.'
!pip install -q -e ./learned_primal_dual 2>/dev/null || echo 'Package already installed.'

In [ ]:
import os
import sys
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from lpd_jax.tomo.geometry import make_parallel_beam_geometry
from lpd_jax.models.learned_primal_dual import LearnedPrimalDual
from lpd_jax.training.data import generate_batch
from lpd_jax.training.train import create_train_state, make_train_step
from lpd_jax.training.evaluate import evaluate
from lpd_jax.tomo.radon import (
  make_batched_radon_forward,
  radon_forward,
  radon_adjoint,
)
from lpd_jax.tomo.opnorm import power_method_opnorm

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

## 1. Geometry and Forward Operator
We define a parallel beam geometry with 128x128 image resolution and 30 projection angles.
Crucially, we compute the **Operator Norm** using the power method to ensure our learned blocks are stable.

In [ ]:
geom = make_parallel_beam_geometry(
  img_shape=(128, 128), img_extent=(128.0, 128.0), num_angles=30
)

print(f"Geometry initialized: {geom.img_shape} images, {geom.num_angles} angles.")

# Compute operator norm for scaling
opnorm = power_method_opnorm(
  lambda x: radon_forward(x, geom),
  lambda y: radon_adjoint(y, geom),
  geom.img_shape,
  num_iter=20,
)
print(f"Calculated Operator Norm: {opnorm:.4f}")

# Define normalized forward operator
_fwd_op = make_batched_radon_forward(geom)


def fwd_op(x: jnp.ndarray) -> jnp.ndarray:
  return _fwd_op(x) / opnorm

## 2. Model Initialization
We initialize the `LearnedPrimalDual` model with 10 iterations. Each iteration "unrolls" a primal mapping and a dual mapping, alternating between image space and measurement space.

In [ ]:
model = LearnedPrimalDual(geometry=geom, n_iter=10, op_norm=opnorm)
rng = jax.random.PRNGKey(42)
rng, init_rng = jax.random.split(rng)

# Dummy input to initialize parameters
dummy_y = jnp.ones((1, geom.num_angles, geom.det_count, 1))
state = create_train_state(init_rng, model, dummy_y, init_lr=1e-3, decay_steps=100000)
train_step = make_train_step()

total_params = sum(x.size for x in jax.tree_util.tree_leaves(state.params))
print(f"Model initialized. Total parameters: {total_params:,}")

## 3. Dataset Generation
We pre-generate 1000 training samples (each with batch_size=8) so the
training loop can iterate through them without per-step data generation overhead.

In [ ]:
N_SAMPLES = 1000
BATCH_SIZE = 8

print(f"Pre-generating {N_SAMPLES} training samples (batch_size={BATCH_SIZE})...")
dataset_y = []
dataset_x = []
for i in range(N_SAMPLES):
  rng, batch_rng = jax.random.split(rng)
  seed = int(jax.random.randint(batch_rng, (), 0, 1000000))
  y_batch, x_true_batch = generate_batch(
    geom, batch_size=BATCH_SIZE, rng_seed=seed, fwd_op=fwd_op
  )
  dataset_y.append(y_batch)
  dataset_x.append(x_true_batch)
  if (i + 1) % 200 == 0:
    print(f"  Generated {i + 1}/{N_SAMPLES} samples.")

print("Dataset ready.")

## 4. Training Loop
We iterate over the pre-generated dataset for several epochs, shuffling
the sample order each epoch.

In [ ]:
N_EPOCHS = 10

print(f"Starting Training ({N_EPOCHS} epochs, {N_SAMPLES} samples each)...")
global_step = 0
for epoch in range(1, N_EPOCHS + 1):
  # Shuffle dataset order each epoch.
  rng, perm_rng = jax.random.split(rng)
  perm = jax.random.permutation(perm_rng, N_SAMPLES)

  epoch_loss = 0.0
  for i in range(N_SAMPLES):
    idx = int(perm[i])
    y_batch = dataset_y[idx]
    x_true_batch = dataset_x[idx]

    state, loss = train_step(state, y_batch, x_true_batch)
    epoch_loss += float(loss)
    global_step += 1

  avg_loss = epoch_loss / N_SAMPLES
  print(f"Epoch {epoch:2d}/{N_EPOCHS} | Avg MSE Loss: {avg_loss:.5f}")

## 5. Evaluation and Visualization
Finally, we generate a validation sample and compare the ground truth phantom with the LPD reconstruction.

In [ ]:
# Generate a clean test sample
y_test, x_true = generate_batch(
  geom, batch_size=1, rng_seed=99, validation=True, fwd_op=fwd_op
)

# Reconstruction
x_pred = state.apply_fn({"params": state.params}, y_test)

# Metrics
loss_val, psnr_val = evaluate(state, y_test, x_true)
print(f"Validation PSNR: {float(psnr_val):.2f} dB")

### Comparison Plots
We visualize the original phantom, the sparse-view sinogram (which is what the model sees),
and the reconstructed result.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Plot settings
im_kwargs = {"cmap": "gray", "origin": "lower"}

# 1. Ground Truth
axes[0].imshow(x_true[0, ..., 0], **im_kwargs)
axes[0].set_title("Ground Truth (Ellipses)")
axes[0].axis("off")

# 2. Sinogram
axes[1].imshow(y_test[0, ..., 0], cmap="viridis", aspect="auto", origin="lower")
axes[1].set_title("Input Sinogram (30 angles)")
axes[1].axis("off")

# 3. LPD Reconstruction
axes[2].imshow(x_pred[0, ..., 0], **im_kwargs)
axes[2].set_title(f"LPD Reconstruction\nPSNR: {float(psnr_val):.2f} dB")
axes[2].axis("off")

# 4. Error Map
error = jnp.abs(x_true[0, ..., 0] - x_pred[0, ..., 0])
im_err = axes[3].imshow(error, cmap="hot", origin="lower")
axes[3].set_title("Absolute Error Map")
axes[3].axis("off")
plt.colorbar(im_err, ax=axes[3], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()